In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [4]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [5]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([6], device='cuda:0')


In [6]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


In [7]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


In [8]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [9]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.2564,  0.1766,  0.7732,  0.3006,  0.0254,  0.7578,  0.5693, -0.2451,
          0.5045, -0.1774, -0.5893,  0.0853,  0.0242, -0.6924,  0.3232, -0.1023,
         -0.1628, -1.1089, -0.5178,  0.2291],
        [-0.0322,  0.2993,  0.2002,  0.5366, -0.1455,  0.6248,  0.4722, -0.1645,
          0.0773, -0.0665, -0.6821, -0.2489, -0.1061, -0.6343,  0.0442, -0.3402,
         -0.5595, -0.6311, -0.6860,  0.2016],
        [ 0.1507,  0.2630,  0.3118,  0.1404, -0.0771,  0.5808,  0.5939, -0.0657,
          0.0017, -0.4968, -0.3191, -0.2444, -0.2795, -0.2644, -0.0215, -0.1956,
         -0.2683, -0.7539, -0.6861,  0.2697]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.2564, 0.1766, 0.7732, 0.3006, 0.0254, 0.7578, 0.5693, 0.0000, 0.5045,
         0.0000, 0.0000, 0.0853, 0.0242, 0.0000, 0.3232, 0.0000, 0.0000, 0.0000,
         0.0000, 0.2291],
        [0.0000, 0.2993, 0.2002, 0.5366, 0.0000, 0.6248, 0.4722, 0.0000, 0.0773,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.04

In [10]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

In [12]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

In [13]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0043,  0.0222,  0.0211,  ..., -0.0126, -0.0155,  0.0129],
        [ 0.0275,  0.0058,  0.0203,  ...,  0.0058, -0.0018,  0.0218]],
       device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([-0.0005, -0.0010], device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0412,  0.0246, -0.0262,  ...,  0.0159,  0.0236, -0.0071],
        [-0.0076,  0.0072,  0.0380,  ...,  0.0225,  0.0188, -0.0322]],
       device='cuda:0', grad_fn=<Sl